In [34]:
from pathlib import Path

import duckdb
import pandas as pd

In [35]:
# ============================================================
# CAMINHOS DO PROJETO
# ============================================================

if Path.cwd().name == "src":
    BASE = Path.cwd().parent
else:
    BASE = Path.cwd()

PASTA_GOLD = BASE / "data" / "gold"
PASTA_RESULTADOS = BASE / "resultados"

# ============================================================
# ARQUIVOS DA CAMADA GOLD
# ============================================================

ARQUIVO_FATO = (
    PASTA_GOLD / "fato_passada.parquet"
)

ARQUIVO_PARTICIPANTE = (
    PASTA_GOLD / "dim_participante.parquet"
)

ARQUIVO_PROTOCOLO = (
    PASTA_GOLD / "dim_protocolo.parquet"
)


# ============================================================
# CAMINHOS PARA USAR NO SQL
# ============================================================

fato = ARQUIVO_FATO.as_posix()

participantes = ARQUIVO_PARTICIPANTE.as_posix()

protocolos = ARQUIVO_PROTOCOLO.as_posix()


In [36]:
# ============================================================
# CONEXÃO COM DUCKDB
# ============================================================

conexao = duckdb.connect()

In [37]:
# ============================================================
# FUNÇÃO PARA EXECUTAR UM TESTE
# ============================================================

#A consulta deve retornar a quantidade de registros que violam a regra.
def executar_teste(nome, tipo, consulta):      

    resultado = conexao.execute(
        consulta
    ).fetchone()

    violacoes = resultado[0]

    if violacoes == 0:
        status = "PASSOU"
    else:
        status = "FALHOU"

    return {
        "teste": nome,
        "tipo": tipo,
        "violacoes": violacoes,
        "status": status
    }


# ============================================================
# LISTA DE RESULTADOS
# ============================================================

resultados = []

In [38]:
# ============================================================
# TESTE 1 - UNICIDADE
# cada participante_key deve identificar uma única pessoa.
# ============================================================

consulta = f"""
SELECT
    COUNT(*) - COUNT(DISTINCT participante_key)
        AS violacoes
FROM read_parquet('{participantes}');
"""

resultados.append(
    executar_teste(
        "participante_key unica",
        "unicidade",
        consulta
    )
)

In [39]:
# ============================================================
# TESTE 2 - NULO
# intervalo_passada_(s) não pode ser nulo
# ============================================================

consulta = f"""
SELECT
    COUNT(*) AS violacoes
FROM read_parquet('{fato}')
WHERE "intervalo_passada_(s)" IS NULL;
"""

resultados.append(
    executar_teste(
        "intervalo_passada_(s) nao nulo",
        "nulo",
        consulta
    )
)

In [40]:
# =====================================================================
# TESTE 3 - DOMÍNIO
# grupo só pode conter categorias conhecidas (Idosos/Jovens/Parkinson)
# =====================================================================

consulta = f"""
SELECT
    COUNT(*) AS violacoes
FROM read_parquet('{participantes}')
WHERE grupo IS NULL
   OR grupo NOT IN (
        'jovem',
        'idoso',
        'parkinson'
   );
"""

resultados.append(
    executar_teste(
        "grupo com valores validos",
        "dominio",
        consulta
    )
)

In [41]:
# ============================================================
# TESTE 4 - DOMÍNIO
# intervalo de passada deve ser positivo
# ============================================================

consulta = f"""
SELECT
    COUNT(*) AS violacoes
FROM read_parquet('{fato}')
WHERE "intervalo_passada_(s)" <= 0;
"""

resultados.append(
    executar_teste(
        "intervalo_passada_(s) positivo",
        "dominio",
        consulta
    )
)

In [42]:
# ============================================================
# TESTE 5 - CHAVE ESTRANGEIRA
# participante da Tabela Fato precisa existir na dimensão
# ============================================================

consulta = f"""
SELECT
    COUNT(*) AS violacoes

FROM read_parquet('{fato}') AS f

LEFT JOIN read_parquet('{participantes}') AS p
    ON f.participante_key = p.participante_key

WHERE p.participante_key IS NULL;
"""

resultados.append(
    executar_teste(
        "participante existente na dimensao",
        "chave estrangeira",
        consulta
    )
)

In [43]:
# ============================================================
# TESTE 6 - CHAVE ESTRANGEIRA
# protocolo da Tabela Fato precisa existir na dimensão
# ============================================================

consulta = f"""
SELECT
    COUNT(*) AS violacoes

FROM read_parquet('{fato}') AS f

LEFT JOIN read_parquet('{protocolos}') AS p
    ON f.protocolo_key = p.protocolo_key

WHERE p.protocolo_key IS NULL;
"""

resultados.append(
    executar_teste(
        "protocolo existente na dimensao",
        "chave estrangeira",
        consulta
    )
)

In [44]:
# ================================================================
# TESTE 7 - REGRA DE COMPLETUDE
# idade deve seguir a documentação do dataset (Parkinson = NULL)
# ================================================================

consulta = f"""
SELECT
    COUNT(*) AS violacoes
FROM read_parquet('{participantes}')
WHERE
    (
        grupo IN ('jovem', 'idoso')
        AND idade IS NULL
    )
    OR
    (
        grupo = 'parkinson'
        AND idade IS NOT NULL
    );
"""

resultados.append(
    executar_teste(
        "idade coerente com documentacao",
        "completude",
        consulta
    )
)

In [71]:
# ============================================================
# RESULTADOS
# ============================================================

tabela_resultados = pd.DataFrame(
    resultados
)

print()
print("=" * 100)
print("RELATÓRIO DOS TESTES DE QUALIDADE")
print("=" * 100)

print(
    tabela_resultados.to_string(
        index=False,
          col_space={
            "teste": 45,
            "tipo": 22,
            "violacoes": 15,
            "status": 12
        },
        justify="justify-all"
    )
)


RELATÓRIO DOS TESTES DE QUALIDADE
                                        teste                   tipo       violacoes       status
                       participante_key unica              unicidade               0       PASSOU
               intervalo_passada_(s) nao nulo                   nulo               0       PASSOU
                    grupo com valores validos                dominio               0       PASSOU
               intervalo_passada_(s) positivo                dominio               0       PASSOU
           participante existente na dimensao      chave estrangeira               0       PASSOU
              protocolo existente na dimensao      chave estrangeira               0       PASSOU
              idade coerente com documentacao             completude               0       PASSOU
